# Cell 1: Imports and path setup

In [1]:
import sys
import os
import time
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# Make ``src`` importable when running from the ``notebooks/`` directory.
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.retriever import Retriever, create_retriever_callable
from src.agents.lamer import LameRAgent

c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cell 2: Configuration

In [2]:
load_dotenv(PROJECT_ROOT / ".env")

class Config:
    # Data paths
    QUERIES_PATH = PROJECT_ROOT / "notebooks" / "queries" / "topics.ms-marco-dev2.tsv"
    QRELS_PATH = PROJECT_ROOT / "notebooks" / "qrels" / "qrels.ms-marco-dev2.tsv"

    # Evaluation scope
    NUM_QUERIES = 20          # Set to an int (e.g. 50) to evaluate a subset.
    NDCG_K = 50                 # LameR paper often reports nDCG@10.
    RECALL_K = 100              # LameR paper reports Recall@1000; use 100 for quick tests.

    # Agent hyperparameters
    N_CANDIDATES = 5
    TOP_K_INITIAL = 20          # Passages shown to the LLM.
    TOP_K_FINAL = 50            # Final BM25 window.

    # Output
    OUTPUT_DIR = PROJECT_ROOT / "outputs"
    OUTPUT_CSV = OUTPUT_DIR / "lamer_isolation_results.csv"


cfg = Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Shared embedding model (required by AgentBase, not used by LameR itself).
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
LLM_NAME = "google/gemma-4-E4B-it"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4720.01it/s]


# Cell 3: List currently available HPC models


In [7]:
import json
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

MODELS_URL = "https://hub.nhr.fau.de/api/llmgw/v1/models"
api_key = os.getenv("LLMAPI_KEY")

if not api_key:
    raise RuntimeError("LLMAPI_KEY is not set; cannot fetch available models.")

resp = requests.get(
    MODELS_URL,
    headers={"Authorization": f"Bearer {api_key}"},
    verify=False,
    timeout=30,
)
resp.raise_for_status()

models_data = resp.json()
print(f"Fetched {len(models_data)} models from {MODELS_URL}")

# Display as a table if the response is a list of dicts; otherwise raw.
if isinstance(models_data, list) and models_data and isinstance(models_data[0], dict):
    display(pd.DataFrame(models_data))
else:
    print(json.dumps(models_data, indent=2))


Fetched 2 models from https://hub.nhr.fau.de/api/llmgw/v1/models
{
  "data": [
    {
      "id": "llamaindex/vdr-2b-multi-v1",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "google/gemma-4-E4B-it",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "deepseek-ai/DeepSeek-V4-Flash",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "MiniMaxAI/MiniMax-M3-MXFP8",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "moonshotai/Kimi-K2.6",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "RedHatAI/Mistral-Small-3.2-24B-Instruct-2506-FP8",
      "object": "model",
      "created": 1677610602,
      "owned_by": "openai"
    },
    {
      "id": "ibm-granite/granite-4.1-3b",
      "object": "model",
      "created": 1

# Cell 4: Data loading helpers

In [3]:
def load_qrels(qrels_path: Path) -> Dict[str, Dict[str, int]]:
    """Load qrels as ``{query_id: {doc_id: relevance_grade}}``."""
    qrels = defaultdict(dict)
    if not qrels_path.exists():
        raise FileNotFoundError(f"Qrels file not found: {qrels_path}")

    with open(qrels_path, "r", encoding="utf-8") as f:
        next(f, None)  # Skip header.
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 4:
                continue
            query_id, doc_id, grade_str = parts[0].strip(), parts[2].strip(), parts[3].strip()
            try:
                grade = int(grade_str)
            except ValueError:
                continue
            qrels[query_id][doc_id] = grade
    return dict(qrels)


def load_queries(queries_path: Path, num_queries: int = None) -> List[Tuple[str, str]]:
    """Load queries as ``[(query_id, query_text), ...]``."""
    queries = []
    if not queries_path.exists():
        raise FileNotFoundError(f"Queries file not found: {queries_path}")

    with open(queries_path, "r", encoding="utf-8") as f:
        next(f, None)  # Skip header.
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) >= 2:
                query_id, query_text = parts[0].strip(), parts[1].strip()
            else:
                query_id, query_text = str(len(queries)), parts[0].strip()
            queries.append((query_id, query_text))
            if num_queries is not None and len(queries) >= num_queries:
                break
    return queries

# Cell 5: Metric helpers (mirror Simulation.compute_ndcg / compute_recall)

In [4]:
def _dcg(relevances: np.ndarray, k: int) -> float:
    relevances = np.asarray(relevances, dtype=float)[:k]
    if relevances.size == 0:
        return 0.0
    positions = np.arange(2, relevances.size + 2)
    return float(np.sum(relevances / np.log2(positions)))


def normalize_doc_id(doc_id: str) -> str:
    """Strip segment suffix (e.g. 'doc#1' -> 'doc') to match qrels format."""
    return doc_id.split("#", 1)[0] if "#" in doc_id else doc_id


def deduplicate_doc_ids(doc_ids: List[str]) -> List[str]:
    """Normalize then deduplicate doc IDs, matching Simulation.deduplicate_doc_ids."""
    deduped = []
    seen = set()
    for doc_id in doc_ids:
        normalized = normalize_doc_id(doc_id)
        if normalized not in seen:
            deduped.append(normalized)
            seen.add(normalized)
    return deduped


def compute_ndcg(ranked_doc_ids: List[str], qrels: Dict[str, int], k: int = 10) -> float:
    ranked_docs = deduplicate_doc_ids(ranked_doc_ids)[:k]
    gains = [qrels.get(doc_id, 0) for doc_id in ranked_docs]
    ideal = sorted((rel for rel in qrels.values() if rel > 0), reverse=True)[:k]
    dcg = _dcg(np.array(gains, dtype=float), k)
    idcg = _dcg(np.array(ideal, dtype=float), k)
    return dcg / idcg if idcg > 0 else 0.0


def compute_recall(ranked_doc_ids: List[str], qrels: Dict[str, int], k: int = 100) -> float:
    ranked_docs = deduplicate_doc_ids(ranked_doc_ids)[:k]
    relevant = {d for d, r in qrels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(set(ranked_docs) & relevant) / len(relevant)

# Cell 6: Initialize retriever and LameR agent

In [5]:
retriever_instance = Retriever(
    endpoint=os.getenv("RETRIEVAL_ENDPOINT"),
    username=os.getenv("MY_USERNAME"),
    password=os.getenv("MY_PASSWORD"),
    index_field="segment",
    top_k=cfg.TOP_K_FINAL,
)
retriever_func = create_retriever_callable(retriever_instance)

lamer_agent = LameRAgent(
    embed_model=EMBED_MODEL,
    n_candidates=cfg.N_CANDIDATES,
    top_k_initial=cfg.TOP_K_INITIAL,
    top_k_final=cfg.TOP_K_FINAL,
    model_name= LLM_NAME
)

[LameR] LLM client config: base_url=https://hub.nhr.fau.de/api/llmgw/v1, model=google/gemma-4-E4B-it


# Cell 7: Load data

In [6]:
queries = load_queries(cfg.QUERIES_PATH, num_queries=cfg.NUM_QUERIES)
qrels = load_qrels(cfg.QRELS_PATH)

print(f"Loaded {len(queries)} queries.")
print(f"Loaded qrels for {len(qrels)} queries.")

Loaded 20 queries.
Loaded qrels for 5000 queries.


# Cell 8: Run isolated evaluation

In [7]:
# %% Cell 7: Run isolated evaluation and cache rankings
import json

records = []

for query_id, query_text in queries:
    print(f"[{len(records)+1}/{len(queries)}] Query {query_id}: {query_text[:60]}...")
    # Baseline BM25
    bm25_start = time.time()
    bm25_doc_ids, bm25_scores, _ = retriever_func(query_text, cfg.TOP_K_FINAL)
    bm25_elapsed = time.time() - bm25_start

    # LameR augmentation + re-retrieval
    effects = lamer_agent.compute_effects({
        "query_text": query_text,
        "retriever": retriever_func,
        "top_k": cfg.TOP_K_FINAL,
    })

    lamer_doc_ids = effects["new_doc_ids"]
    lamer_elapsed = effects["elapsed_time"]
    augmented_query = effects["new_query_text"]

    qrels_for_query = qrels.get(query_id, {})

    # Metrics
    bm25_ndcg = compute_ndcg(bm25_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    lamer_ndcg = compute_ndcg(lamer_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    bm25_recall = compute_recall(bm25_doc_ids, qrels_for_query, k=cfg.RECALL_K)
    lamer_recall = compute_recall(lamer_doc_ids, qrels_for_query, k=cfg.RECALL_K)

    records.append({
        "query_id": query_id,
        "query_text": query_text,
        "augmented_query": augmented_query,
        "bm25_doc_ids": ";".join(bm25_doc_ids),
        "lamer_doc_ids": ";".join(lamer_doc_ids),
        "bm25_ndcg": bm25_ndcg,
        "lamer_ndcg": lamer_ndcg,
        "ndcg_gain": lamer_ndcg - bm25_ndcg,
        "bm25_recall": bm25_recall,
        "lamer_recall": lamer_recall,
        "recall_gain": lamer_recall - bm25_recall,
        "bm25_latency_ms": bm25_elapsed * 1000,
        "lamer_latency_ms": lamer_elapsed * 1000,
        "augmented_extra_tokens": len(augmented_query.split()) - len(query_text.split()),
    })

# Save enriched cache
df = pd.DataFrame(records)
df.to_csv(cfg.OUTPUT_CSV, index=False)
print(f"Saved rankings + metrics to: {cfg.OUTPUT_CSV}")

[1/20] Query 1048579: what is pcnt...


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[LameR] Final candidates (5): ['PCNT can stand for several things, including Pericentrin, Panama Canal Net Tonnage, or Percutaneous Nephrostomy Tube, depending on the context.', 'PCNT can refer to several things, including Pericentrin in biology, Panama Canal Net Tonnage in shipping, or Percutaneous Nephrostomy Tube in urology.', 'PCNT can stand for several things, including Pericentrin (a protein crucial for cell division), Panama Canal Net Tonnage (in shipping), or Percutaneous Nephrostomy Tube (in.', 'PCNT commonly stands for Pericentrin, which is a protein located in centrosomes and plays a crucial role in cell division.', 'PCNT is an abbreviation that can stand for Pericentrin, a centrosomal protein crucial for cell division, or Panama Canal Net Tonnage in shipping.']


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[2/20] Query 262156: how long is a college hockey game...


...

[20/20] Query 43873: average settlement for hearing damage and loss...


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


[LameR] Final candidates (5): ['The average payout in hearing injury malpractice cases where the defendant was an otolaryngologist was $313,230.', 'The average payout in hearing injury malpractice cases where an otolaryngologist was the defendant was $313,230.', 'The average payout in cases where an otolaryngologist was the defendant for a hearing injury was $313,230.', 'The average payout in hearing injury malpractice cases sued against otolaryngologists was $313,230.', 'The average payout in cases where otolaryngologists were the defendant for a hearing injury was $313,230.']


c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'opensearch.pads.fim.uni-passau.de'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Saved rankings + metrics to: c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\outputs\lamer_isolation_results.csv


# Cell 9-alt: Recompute metrics from cached rankings + NEW qrels

In [29]:
# Load the cached results (produced by Cell 7 above)
df_cached = pd.read_csv(cfg.OUTPUT_CSV)

# Load the CORRECT qrels (change path here if needed)
CORRECT_QRELS_PATH = cfg.QRELS_PATH  # or override: Path("data/qrels/correct.qrels.tsv")
correct_qrels = load_qrels(CORRECT_QRELS_PATH)

records = []

for _, row in df_cached.iterrows():
    query_id = row["query_id"]
    qrels_for_query = correct_qrels.get(query_id, {})

    bm25_doc_ids = row["bm25_doc_ids"].split(";") if pd.notna(row["bm25_doc_ids"]) else []
    lamer_doc_ids = row["lamer_doc_ids"].split(";") if pd.notna(row["lamer_doc_ids"]) else []

    bm25_ndcg = compute_ndcg(bm25_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    lamer_ndcg = compute_ndcg(lamer_doc_ids, qrels_for_query, k=cfg.NDCG_K)
    bm25_recall = compute_recall(bm25_doc_ids, qrels_for_query, k=cfg.RECALL_K)
    lamer_recall = compute_recall(lamer_doc_ids, qrels_for_query, k=cfg.RECALL_K)

    records.append({
        "query_id": query_id,
        "query_text": row["query_text"],
        "augmented_query": row["augmented_query"],
        "bm25_ndcg": bm25_ndcg,
        "lamer_ndcg": lamer_ndcg,
        "ndcg_gain": lamer_ndcg - bm25_ndcg,
        "bm25_recall": bm25_recall,
        "lamer_recall": lamer_recall,
        "recall_gain": lamer_recall - bm25_recall,
        "bm25_latency_ms": row["bm25_latency_ms"],
        "lamer_latency_ms": row["lamer_latency_ms"],
        "augmented_extra_tokens": row["augmented_extra_tokens"],
    })

df = pd.DataFrame(records)
df.to_csv(cfg.OUTPUT_CSV, index=False)
print(f"Overwrote results with corrected qrels: {cfg.OUTPUT_CSV}")

KeyError: 'bm25_doc_ids'

# Cell 9: Summarize and save

In [8]:
df = pd.DataFrame(records)
df.to_csv(cfg.OUTPUT_CSV, index=False)

print(f"\nSaved per-query results to: {cfg.OUTPUT_CSV}")
print(f"Evaluated queries: {len(df)}")
print(f"Queries where at least one retrieval method retrieved a relevant document: {(df['bm25_ndcg'] + df['lamer_ndcg'] > 0).sum()}")

print("\n=== Overall Averages ===")
print(f"BM25  nDCG@{cfg.NDCG_K}:     {df['bm25_ndcg'].mean():.4f}")
print(f"LameR nDCG@{cfg.NDCG_K}:     {df['lamer_ndcg'].mean():.4f}")
print(f"Mean nDCG gain:              {df['ndcg_gain'].mean():+.4f}")
print(f"Win rate (LameR > BM25):     {(df['ndcg_gain'] > 0).mean():.1%}")

print(f"\nBM25  Recall@{cfg.RECALL_K}:   {df['bm25_recall'].mean():.4f}")
print(f"LameR Recall@{cfg.RECALL_K}:   {df['lamer_recall'].mean():.4f}")
print(f"Mean Recall gain:            {df['recall_gain'].mean():+.4f}")

print(f"\nBM25  latency: {df['bm25_latency_ms'].mean():.1f} ms/query")
print(f"LameR latency: {df['lamer_latency_ms'].mean():.1f} ms/query")

# %% Cell 9: Top winners / losers by nDCG gain
print("\n=== Top 10 nDCG gains ===")
print(df.sort_values("ndcg_gain", ascending=False)[[
    "query_id", "query_text", "ndcg_gain", "bm25_ndcg", "lamer_ndcg"
]].head(10).to_string(index=False))

print("\n=== Top 10 nDCG losses ===")
print(df.sort_values("ndcg_gain", ascending=True)[[
    "query_id", "query_text", "ndcg_gain", "bm25_ndcg", "lamer_ndcg"
]].head(10).to_string(index=False))


Saved per-query results to: c:\Users\hanaz\Documents\GitHub\Multi-Agent-Ensemble-for-Search-Through-Reinforcement-Optimization-MAESTRO-\outputs\lamer_isolation_results.csv
Evaluated queries: 20
Queries where at least one retrieval method retrieved a relevant document: 11

=== Overall Averages ===
BM25  nDCG@50:     0.3275
LameR nDCG@50:     0.3324
Mean nDCG gain:              +0.0049
Win rate (LameR > BM25):     20.0%

BM25  Recall@100:   0.5250
LameR Recall@100:   0.4500
Mean Recall gain:            -0.0750

BM25  latency: 915.9 ms/query
LameR latency: 4328.8 ms/query

=== Top 10 nDCG gains ===
query_id                                      query_text  ndcg_gain  bm25_ndcg  lamer_ndcg
  524574                          trending topic meaning   0.698970   0.301030    1.000000
 1048779                               what is ott media   0.412825   0.218104    0.630930
 1048601                       what is pastoral medicine   0.200253   0.430677    0.630930
  786568  what is price of press

In [3]:
import pandas as pd
df_lamer = pd.read_csv("../archive/legacy_outputs/lamer_isolation_results.csv")
df_lamer

,query_id,query_text,augmented_query,bm25_doc_ids,lamer_doc_ids,bm25_ndcg,lamer_ndcg,ndcg_gain,bm25_recall,lamer_recall,recall_gain,bm25_latency_ms,lamer_latency_ms,augmented_extra_tokens
0,1048579,what is pcnt,what is pcnt PCNT can stand for several things...,msmarco_v2.1_doc_32_822435716#0_1580718467;msm...,msmarco_v2.1_doc_32_822435716#0_1580718467;msm...,1.000000,1.000000,0.000000,1.0,1.0,0.0,1321.913004,8067.784071,122
1,262156,how long is a college hockey game,how long is a college hockey game A college ho...,msmarco_v2.1_doc_03_1606070005#9_2735961893;ms...,msmarco_v2.1_doc_22_285492708#3_696718122;msma...,0.630930,0.500000,-0.130930,1.0,1.0,0.0,861.791372,3190.081596,37
2,1048601,what is pastoral medicine,what is pastoral medicine Pastoral Science and...,msmarco_v2.1_doc_48_1032635938#0_1860247067;ms...,msmarco_v2.1_doc_49_1332145331#6_2768423002;ms...,0.430677,0.630930,0.200253,1.0,1.0,0.0,1145.263433,4431.708097,86
3,1048673,what is ownership of a corporation called,what is ownership of a corporation called The ...,msmarco_v2.1_doc_47_1066242117#0_2304582256;ms...,msmarco_v2.1_doc_56_743855141#0_1526747263;msm...,0.000000,0.000000,0.000000,0.0,0.0,0.0,1207.329750,1689.033985,8
4,786531,what is prevail,what is prevail To prevail means to predominat...,msmarco_v2.1_doc_07_786991500#2_1386662516;msm...,msmarco_v2.1_doc_53_880012450#5_1940168178;msm...,0.000000,0.000000,0.000000,0.0,0.0,0.0,621.627569,8462.893248,61
5,1048706,what is overhead rate in managerial accounting?,what is overhead rate in managerial accounting...,msmarco_v2.1_doc_00_81958265#0_151967731;msmar...,msmarco_v2.1_doc_14_1157132730#1_2410641639;ms...,0.000000,0.000000,0.000000,0.0,0.0,0.0,779.320717,4530.749798,85
6,786568,what is price of pressure treated lumber 2x6x8,what is price of pressure treated lumber 2x6x8...,msmarco_v2.1_doc_06_1650884164#16_2409386164;m...,msmarco_v2.1_doc_06_1650884164#16_2409386164;m...,0.356207,0.386853,0.030646,1.0,1.0,0.0,778.620958,5266.956806,71
7,1048730,what is outlook data file,what is outlook data file An Outlook data file...,msmarco_v2.1_doc_52_1302064938#0_2628177208;ms...,msmarco_v2.1_doc_49_226935335#0_445923186;msma...,0.000000,0.000000,0.000000,0.0,0.0,0.0,1350.970507,3432.470322,78
8,262330,how long is a flight from chicago to australia,how long is a flight from chicago to australia...,msmarco_v2.1_doc_10_137501932#0_260323258;msma...,msmarco_v2.1_doc_10_137501932#0_260323258;msma...,1.000000,1.000000,0.000000,1.0,1.0,0.0,869.732141,5052.935839,93
9,1048779,what is ott media,what is ott media OTT media refers to media di...,msmarco_v2.1_doc_37_564669280#3_1211491422;msm...,msmarco_v2.1_doc_37_564669280#4_1211493443;msm...,0.218104,0.630930,0.412825,1.0,1.0,0.0,773.436069,4524.931908,71
